In [11]:
from bs4 import BeautifulSoup as beau
import pandas as pd
import itertools
import pickle
import nltk
import os
import re
import dask
from dask.distributed import Client, LocalCluster
client = Client(processes=True, n_workers=2, threads_per_worker=8)

c:\Users\jbean\anaconda3\Lib\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 50092 instead
  warnings.warn(


In [12]:
fics_df = pd.DataFrame(columns=['L1', 'author', 'title', 'chapters', 'work', 'summary', 'notes', 'endnotes', 'rating', 'warnings', 'ships', 'characters', 'freeform', 'fandoms', 'word_count', 'punct'])
r_rating = re.compile(r'Rating:</dt>\n<dd>.*?</dd>', re.S)
r_warnings = re.compile(r'Warnings*:</dt>\n<dd>.*?</dd>', re.S)
r_fandoms = re.compile(r'Fandoms*:</dt>\n<dd>.*?</dd>', re.S)
r_ships = re.compile(r'Relationships*:</dt>\n<dd>.*?</dd>', re.S)
r_characters = re.compile(r'Characters*:</dt>\n<dd>.*?</dd>', re.S)
r_freeform = re.compile(r'Additional Tags:</dt>\n<dd>.*?</dd>', re.S)
r_summary = re.compile(r'(?<=Summary</p>).*?Notes', re.S)
r_notes = re.compile(r'(?<=Notes</p>).+', re.S)
r_endnotes = re.compile(r'(?<=End Notes).*?</blockquote', re.S)
notes_clean = re.compile(r'<p>.*?(?=</blockquote)', re.S)
work_a = re.compile(r'(?<=<!--chapter content-->).*?<!--/chapter content-->', re.S)
work_b = re.compile(r'.*?(?=<!--/chapter content-->)', re.S)

In [13]:
r_title = re.compile(r'(?<=<h2 class="title heading">).*?</', re.S)
r_chapters = re.compile(r'(?<=Chapters:</dt><dd class="chapters">)\d*/\d*', re.S)
r_words = re.compile(r'(?<=Words:)\d[\d,]*', re.S)

In [14]:
def get_work(s): 
    s = re.sub(r'[^\S\n]+', ' ', s)
    cc = re.compile(r'<!--chapter content-->.*?<!--/chapter content-->', re.S)
    emm = re.compile(r'\n<span>', re.S)
    emmm = re.compile(r'</span>\n', re.S)
    p = re.compile(r'</p><p>')
    c = re.findall(cc, s)
    c = [re.sub(p, '\n', re.sub(emmm, '', re.sub(r'</span>\n*', '', re.sub(r'\n*<span>', '', re.sub(r'^\n', '', re.sub(r'(\n){3,}', '\n\n\n', re.sub(r'\n \n', '\n', beau(ch).text))))))) for ch in c]
    return c

In [15]:
def get_only_chap(s):
    s = re.sub(r'[^\S\n]+', ' ', s)
    init = re.compile(r'(?<=<div id="chapters" class="userstuff">).*', re.S)
    w = re.compile(r'(?<=<p>).*', re.S)
    cc = re.compile(r'.*(?=</div>\n</div>)', re.S)
    emm = re.compile(r'\n<span>', re.S)
    emmm = re.compile(r'</span>\n', re.S)
    p = re.compile(r'</p><p>')
    c = re.search(cc, s)[0]
    c = re.search(init, c)[0]
    c = re.search(w, c)[0]
    c = re.sub(r'\n \n', '\n', c)
    c = re.sub(r'(\n){3,}', '\n\n\n', c)
    c = re.sub(r'^\n', '', c)
    c = re.sub(r'\n*<span>', '', c)
    c = re.sub(r'</span>\n*', '', c)
    c = re.sub(emm, '', c)
    c = re.sub(emmm, '', c)
    c = re.sub(p, '\n', c)
    return [c]

In [16]:
def get_tags(pat, s):
    lst = []
    a = re.search(pat, str(s))[0]
    b = re.findall(r'(?<=">).*?</a', a)
    for x in b:
        c = re.search(r'.*(?=<)', x)[0]
        lst.append(c)
    return lst

In [17]:
def get_notes(pat, soup):
    s = re.search(pat, str(soup))[0]
    return beau(re.search(notes_clean, s)[0]).text

In [18]:
def get_chap(x):
    return re.search(r'(?<=Chapters: ).*', str(x))[0]

In [19]:
def get_wc(f):
    try:
        x = re.search(r_words, f)[0]
        return convert_wc(re.sub(',', '', x))
    except:
        return re.search(r'(?<=Words:\n)\d*', beau(f).text)[0]
    
def convert_wc(x):
    try:
        return int(x)
    except:
        return None

In [20]:
def get_title(f):
    try:
        return beau(re.search(r_title, f)[0]).text
    except:
        return None

In [21]:
def parse_fic(st):
    soup = beau(st)
    L1 = soup.find(class_='L1').text
    try:
        author = soup.find(rel='author').text
    except:
        author = 'Anonymous'
    title = get_title(st)
    chapters = get_chap(soup)
    try:
        work = get_only_chap(st)
    except:
        work = get_work(st)
    try:
        summary = get_notes(r_summary, soup)
    except:
        summary = None
    try:
        note = get_notes(r_notes, soup)
    except:
        note = None
    try:
        endnote = get_notes(r_endnotes, soup)
    except:
        endnote = None
    rating = get_tags(r_rating, soup)
    try:
        warning = get_tags(r_warnings, soup)
    except:
        warning = None
    try:
        fandom = get_tags(r_fandoms, soup)
    except:
        fandom = None
    try:
        ship = get_tags(r_ships, soup)
    except:
        ship = None
    try:
        chara = get_tags(r_characters, soup)
    except:
        chara = None
    try:
        freeform = get_tags(r_freeform, soup)
    except:
        freeform = None
        try:
            wc = get_wc(soup)
        except:
            wc = 0
    return pd.DataFrame({'L1':[L1], 'author':[author], 'title':[title], 'chapters':[chapters], 'work':[work], 'summary':[summary], 'notes':[note], 'endnotes':[endnote], 'rating':[rating], 'warnings':[warning], 'fandoms':[fandom], 'ships':[ship], 'characters':[chara],'freeform':[freeform], 'word_count':[wc]})

In [23]:
st

'\n\n\n<h3 class="landmark heading">Work Header</h3>\n\n<div class="wrapper">\n\n  <dl class="work meta group">\n          <dt class="rating tags">\n\n              Rating:\n          </dt>\n\n          <dd class="rating tags">\n            <ul class="commas">\n               <li><a class="tag" href="/tags/General%20Audiences/works">General Audiences</a></li>\n            </ul>\n          </dd>\n          <dt class="warning tags">\n\n              <a href="/tos_faq#tags">Archive Warning</a>:\n          </dt>\n\n          <dd class="warning tags">\n            <ul class="commas">\n               <li><a class="tag" href="/tags/No%20Archive%20Warnings%20Apply/works">No Archive Warnings Apply</a></li>\n            </ul>\n          </dd>\n          <dt class="category tags">\n\n              Categories:\n          </dt>\n\n          <dd class="category tags">\n            <ul class="commas">\n               <li><a class="tag" href="/tags/Gen/works">Gen</a></li><li><a class="tag" href="/tags

In [22]:
files = []
directory = 'natfinder/natfinder/manual-works'
for f in os.scandir(directory):
    files.append(f.name)
for f in files:
    st = open(directory + '/' + f).read()
    temp_df = dask.delayed(parse_fic(st))
    fics_df = pd.concat([fics_df, temp_df.compute()], ignore_index=True)

AttributeError: 'NoneType' object has no attribute 'text'

In [ ]:
fics = fics_df
fics['notes'] = fics_df['notes'].fillna('') + fics_df['endnotes'].fillna('')
fics_df['punct'] = fics_df.work.map(lambda x : [z for z in set([y for y in ''.join(x)]) if not z.isalpha() and not z.isdigit()])

In [ ]:
#punctuation = set([y for y in ''.join([''.join(x) for x in fics_df['punct'].values])])
#punct_dict = {'"':'quotation mark', "'":'single quote', '“':'left smart quote', '”':'right smart quote', '‘':'left smart single quote', '’':'right smart single quote', '„':'low left quotation', '«':'left double angle bracket', '»':'right double angle bracket', '‹':'left single angle bracket', '›':'right single angle bracket', '〝':'reversed double ideographic', '〞':'double ideographic', '「':'left corner bracket', '」':'right corner bracket', '『':'left corner bracket bold', '』':'right corner bracket bold', '《':'left double-angle large', '》':'right double-angle large', '〈':'left single-angle large', '〉':'right single-angle large', '‚':'low left single'}
#punct_list = list(punct_dict.keys())
#for x in punct_list:
#    punct_df[x] = ''
## if count(\s\?) > (\S\?)
## if count(\s!) > (\S!)
## dashes (--), (---), (\s-\s), (\S-\s) || (\s-\S), (en dash), (em dash), (\sen dash) || (en dash\s), (\sem dash), (em dash\s), etc
## periods, commas, exclamation marks, question marks, semicolon, dash, colon, parentheses
